# Results Aggregation

This notebook is the final step before writing the report. It pulls together every result file produced by notebooks 03–09 and assembles them into clean tables that can be pasted directly into the report (or into a slide).

**What this notebook produces:**
- A **main results table** comparing all 5 models on in-domain test AUC, accuracy, precision, recall, F1
- A **cross-manipulation table** showing in-domain AUC vs. FaceShifter AUC plus the AUC drop
- A **per-manipulation breakdown table** showing each model's accuracy on each fake type (Deepfakes, Face2Face, FaceSwap, NeuralTextures)
- A **model size / efficiency table** (parameter counts)
- Markdown and LaTeX versions of every table, written to `./results/aggregated/`

The notebook does no training or evaluation. It only reads JSON files that the previous notebooks produced.

### To view TensorBoard
This notebook is just aggregation. So, no TensorBoard logs here.


## 1. Setup

In [1]:
# Import nercessary libraries and the shared utilities
import json
from pathlib import Path
from collections import defaultdict

import numpy as np

from utils import RESULTS_DIR_PATH

In [2]:
# Set the output paths
output_dir_path = RESULTS_DIR_PATH / "aggregated"
output_dir_path.mkdir(parents=True, exist_ok=True)
print(f"Outputs will be saved to: {output_dir_path.resolve()}")

Outputs will be saved to: C:\x-ion\TEST\results\aggregated


## 2. Model registry

The order here is also the order of rows in every table. Baselines first, proposed model last.

In [3]:
# (model_name, display_label, parameter_count_in_millions)
# Parameter counts come from the training notebooks (printed at model-build time).
# If a number isn't known yet just put None and the table will leave the cell blank.
models_in_report_order = [
    {"model_name": "xception_baseline",        "display_label": "Xception",                       "parameter_count_millions": 20.8},
    {"model_name": "efficientnet_b3_baseline",    "display_label": "EfficientNet-B3",                "parameter_count_millions": 10.7},
    {"model_name": "cnn_lstm_baseline",        "display_label": "CNN+BiLSTM",                     "parameter_count_millions": 13.7},
    {"model_name": "vit_base_baseline",             "display_label": "ViT-Base/16",                    "parameter_count_millions": 85.8},
    {"model_name": "hybrid_cnn_transformer",   "display_label": "Hybrid CNN-Transformer (ours)",  "parameter_count_millions": 74.9},
]

# Note: parameter counts above are typical values for these architectures. 
# If the prints in the training notebooks reported different numbers, edit this dict by hand.

## 3. Load every per-model results file

We need three files per model: `final_test_metrics.json` (in-domain test set), `test_predictions.json` (per-sample for the per-manipulation breakdown), and the cross-manipulation file from notebook 08.

In [4]:
# Load in-domain test metrics for every model
in_domain_metrics = {}
for spec in models_in_report_order:
    model_name = spec["model_name"]
    path = RESULTS_DIR_PATH / model_name / "final_test_metrics.json"
    if path.exists():
        with open(path, "r") as f:
            in_domain_metrics[model_name] = json.load(f)
        print(f"Loaded in-domain metrics for: {model_name}")
    else:
        in_domain_metrics[model_name] = None
        print(f"  MISSING: {path}")

Loaded in-domain metrics for: xception_baseline
Loaded in-domain metrics for: efficientnet_b3_baseline
Loaded in-domain metrics for: cnn_lstm_baseline
Loaded in-domain metrics for: vit_base_baseline
Loaded in-domain metrics for: hybrid_cnn_transformer


In [5]:
# Load per-sample test predictions for every model (for per-manipulation breakdown)
in_domain_predictions = {}
for spec in models_in_report_order:
    model_name = spec["model_name"]
    path = RESULTS_DIR_PATH / model_name / "test_predictions.json"
    if path.exists():
        with open(path, "r") as f:
            in_domain_predictions[model_name] = json.load(f)
        print(f"Loaded test predictions for: {model_name}")
    else:
        in_domain_predictions[model_name] = None
        print(f"  MISSING: {path}")

Loaded test predictions for: xception_baseline
Loaded test predictions for: efficientnet_b3_baseline
Loaded test predictions for: cnn_lstm_baseline
Loaded test predictions for: vit_base_baseline
Loaded test predictions for: hybrid_cnn_transformer


In [6]:
# Load cross-manipulation results (this is a single file with all models in it)
cross_manipulation_path = RESULTS_DIR_PATH / "cross_manipulation" / "faceshifter_metrics.json"
if cross_manipulation_path.exists():
    with open(cross_manipulation_path, "r") as f:
        cross_manipulation_payload = json.load(f)
    print(f"Loaded cross-manipulation metrics for: {list(cross_manipulation_payload.keys())}")
else:
    cross_manipulation_payload = {}
    print(f"  MISSING: {cross_manipulation_path}")

Loaded cross-manipulation metrics for: ['xception_baseline', 'efficientnet_b3_baseline', 'cnn_lstm_baseline', 'vit_base_baseline', 'hybrid_cnn_transformer']


## 4. Helper for table formatting

We want both a printed plain-text table (to look at in the notebook) and a markdown + LaTeX version (for the report). One helper that handles all three formats.

In [7]:
def format_value(value, fmt="{:.4f}"):
    """This function formats a value, returning '-' if it is None or NaN."""
    if value is None:
        return "-"
    try:
        f = float(value)
    except (TypeError, ValueError):
        return str(value)
    if np.isnan(f):
        return "-"
    return fmt.format(f)


def render_table(headers, rows, save_basename):
    """This function prints a plain-text table and writes .md and .tex versions."""
    # Plain-text print
    column_widths = [max(len(headers[i]), max((len(r[i]) for r in rows), default=0)) for i in range(len(headers))]
    horizontal_line = "  ".join("-" * w for w in column_widths)
    
    def format_row(values):
        return "  ".join(f"{v:<{column_widths[i]}}" for i, v in enumerate(values))
    
    print(format_row(headers))
    print(horizontal_line)
    for row in rows:
        print(format_row(row))
    
    # Markdown
    md_lines = []
    md_lines.append("| " + " | ".join(headers) + " |")
    md_lines.append("| " + " | ".join("---" for _ in headers) + " |")
    for row in rows:
        md_lines.append("| " + " | ".join(row) + " |")
    md_path = output_dir_path / f"{save_basename}.md"
    md_path.write_text("\n".join(md_lines) + "\n")
    
    # LaTeX (booktabs style)
    column_spec = "l" + "r" * (len(headers) - 1)
    tex_lines = []
    tex_lines.append("\\begin{tabular}{" + column_spec + "}")
    tex_lines.append("\\toprule")
    tex_lines.append(" & ".join(headers) + " \\\\")
    tex_lines.append("\\midrule")
    for row in rows:
        tex_lines.append(" & ".join(row) + " \\\\")
    tex_lines.append("\\bottomrule")
    tex_lines.append("\\end{tabular}")
    tex_path = output_dir_path / f"{save_basename}.tex"
    tex_path.write_text("\n".join(tex_lines) + "\n")
    
    print(f"\nWrote: {md_path}")
    print(f"Wrote: {tex_path}")

## 5. Table 1: Main in-domain results

Every model's performance on the in-domain test split (binary real-vs-fake across all 5 training-class manipulations).

In [8]:
headers = ["Model", "Params (M)", "AUC", "Accuracy", "Precision", "Recall", "F1"]
rows = []
for spec in models_in_report_order:
    payload = in_domain_metrics.get(spec["model_name"])
    if payload is None:
        rows.append([spec["display_label"], format_value(spec["parameter_count_millions"], "{:.1f}"), "-", "-", "-", "-", "-"])
        continue
    m = payload["test_metrics"]
    rows.append([
        spec["display_label"],
        format_value(spec["parameter_count_millions"], "{:.1f}"),
        format_value(m["auc"]),
        format_value(m["accuracy"]),
        format_value(m["precision"]),
        format_value(m["recall"]),
        format_value(m["f1"]),
    ])

render_table(headers, rows, "table1_in_domain")

Model                          Params (M)  AUC     Accuracy  Precision  Recall  F1    
-----------------------------  ----------  ------  --------  ---------  ------  ------
Xception                       20.8        0.9944  0.9744    0.9823     0.9858  0.9840
EfficientNet-B3                10.7        0.9976  0.9829    0.9825     0.9964  0.9894
CNN+BiLSTM                     13.7        0.9805  0.9744    0.9823     0.9858  0.9840
ViT-Base/16                    85.8        0.7066  0.6695    0.8910     0.6690  0.7642
Hybrid CNN-Transformer (ours)  74.9        0.9458  0.8462    0.9710     0.8327  0.8966

Wrote: results\aggregated\table1_in_domain.md
Wrote: results\aggregated\table1_in_domain.tex


## 6. Table 2: Cross-manipulation generalization

In-domain AUC vs. FaceShifter AUC. AUC drop = (in-domain) − (FaceShifter). Smaller drop = better generalization to unseen manipulations.

In [9]:
headers = ["Model", "In-domain AUC", "FaceShifter AUC", "AUC drop", "FaceShifter Acc", "FaceShifter F1"]
rows = []
for spec in models_in_report_order:
    payload = cross_manipulation_payload.get(spec["model_name"])
    if payload is None:
        rows.append([spec["display_label"], "-", "-", "-", "-", "-"])
        continue
    in_domain_auc = payload["in_domain_test_auc"]
    fs_metrics = payload["faceshifter_metrics"]
    auc_drop = in_domain_auc - fs_metrics["auc"] if in_domain_auc is not None else None
    rows.append([
        spec["display_label"],
        format_value(in_domain_auc),
        format_value(fs_metrics["auc"]),
        format_value(auc_drop, "{:+.4f}"),
        format_value(fs_metrics["accuracy"]),
        format_value(fs_metrics["f1"]),
    ])

render_table(headers, rows, "table2_cross_manipulation")

Model                          In-domain AUC  FaceShifter AUC  AUC drop  FaceShifter Acc  FaceShifter F1
-----------------------------  -------------  ---------------  --------  ---------------  --------------
Xception                       0.9944         0.7015           +0.2929   0.3000           0.3160        
EfficientNet-B3                0.9976         0.7343           +0.2633   0.4064           0.4746        
CNN+BiLSTM                     0.9805         0.7591           +0.2214   0.4191           0.4916        
ViT-Base/16                    0.7066         0.6880           +0.0186   0.6170           0.7297        
Hybrid CNN-Transformer (ours)  0.9458         0.6273           +0.3185   0.3404           0.3825        

Wrote: results\aggregated\table2_cross_manipulation.md
Wrote: results\aggregated\table2_cross_manipulation.tex


## 7. Table 3: Per-manipulation accuracy breakdown on in-domain test

For each model, the accuracy on each manipulation type at threshold 0.5. The "Original" column is real-clip accuracy. This shows which fake types are easy or hard for each model.

In [10]:
# Manipulation column order. "original" first, then the four fake training manipulations.
manipulation_column_order = ["original", "Deepfakes", "Face2Face", "FaceSwap", "NeuralTextures"]

headers = ["Model"] + manipulation_column_order
rows = []
for spec in models_in_report_order:
    predictions = in_domain_predictions.get(spec["model_name"])
    if predictions is None:
        rows.append([spec["display_label"]] + ["-"] * len(manipulation_column_order))
        continue
    
    # Group by class_name
    probabilities_by_class = defaultdict(list)
    labels_by_class = defaultdict(list)
    for probability, true_label, class_name in zip(
        predictions["predicted_probs"],
        predictions["true_labels"],
        predictions["class_names"],
    ):
        probabilities_by_class[class_name].append(probability)
        labels_by_class[class_name].append(true_label)
    
    row = [spec["display_label"]]
    for cls_name in manipulation_column_order:
        if cls_name in probabilities_by_class:
            probs = np.array(probabilities_by_class[cls_name])
            true_labels = np.array(labels_by_class[cls_name])
            predicted = (probs >= 0.5).astype(int)
            accuracy = (predicted == true_labels).mean()
            row.append(format_value(accuracy))
        else:
            row.append("-")
    rows.append(row)

render_table(headers, rows, "table3_per_manipulation")

Model                          original  Deepfakes  Face2Face  FaceSwap  NeuralTextures
-----------------------------  --------  ---------  ---------  --------  --------------
Xception                       0.9286    0.9859     0.9857     1.0000    0.9714        
EfficientNet-B3                0.9286    1.0000     1.0000     1.0000    0.9857        
CNN+BiLSTM                     0.9286    1.0000     1.0000     0.9714    0.9714        
ViT-Base/16                    0.6714    0.9014     0.6714     0.5286    0.5714        
Hybrid CNN-Transformer (ours)  0.9000    0.8873     0.8571     0.8143    0.7714        

Wrote: results\aggregated\table3_per_manipulation.md
Wrote: results\aggregated\table3_per_manipulation.tex


## 8. Table 4: Combined summary

A single condensed table that gives the reader the whole story at a glance. We'll likely put this one in the abstract or as Table 1 in the Results section.

In [11]:
headers = ["Model", "Params (M)", "In-domain AUC", "FaceShifter AUC", "AUC drop"]
rows = []
for spec in models_in_report_order:
    in_domain_payload = in_domain_metrics.get(spec["model_name"])
    cross_payload = cross_manipulation_payload.get(spec["model_name"])
    
    in_domain_auc = in_domain_payload["test_metrics"]["auc"] if in_domain_payload is not None else None
    faceshifter_auc = cross_payload["faceshifter_metrics"]["auc"] if cross_payload is not None else None
    auc_drop = (in_domain_auc - faceshifter_auc) if (in_domain_auc is not None and faceshifter_auc is not None) else None
    
    rows.append([
        spec["display_label"],
        format_value(spec["parameter_count_millions"], "{:.1f}"),
        format_value(in_domain_auc),
        format_value(faceshifter_auc),
        format_value(auc_drop, "{:+.4f}"),
    ])

render_table(headers, rows, "table4_combined_summary")

Model                          Params (M)  In-domain AUC  FaceShifter AUC  AUC drop
-----------------------------  ----------  -------------  ---------------  --------
Xception                       20.8        0.9944         0.7015           +0.2929 
EfficientNet-B3                10.7        0.9976         0.7343           +0.2633 
CNN+BiLSTM                     13.7        0.9805         0.7591           +0.2214 
ViT-Base/16                    85.8        0.7066         0.6880           +0.0186 
Hybrid CNN-Transformer (ours)  74.9        0.9458         0.6273           +0.3185 

Wrote: results\aggregated\table4_combined_summary.md
Wrote: results\aggregated\table4_combined_summary.tex


## 9. One JSON file with all aggregated results

Useful for any final cross-checking or for generating extra figures later.

In [12]:
aggregated_payload = {
    "models": models_in_report_order,
    "in_domain_metrics": {k: v for k, v in in_domain_metrics.items() if v is not None},
    "cross_manipulation_metrics": cross_manipulation_payload,
}
aggregated_save_path = output_dir_path / "all_results.json"
with open(aggregated_save_path, "w") as f:
    json.dump(aggregated_payload, f, indent=2)
print(f"Saved aggregated results to: {aggregated_save_path}")

Saved aggregated results to: results\aggregated\all_results.json


## Done

All tables are saved in `./results/aggregated/` as both `.md` and `.tex` files. Copy-paste them straight into the report. The figures from notebook 08 (`auc_comparison.png`) and notebook 09 (`grid_in_domain.png`, `grid_faceshifter.png`) are the visual side of the Results section.

That's the last code notebook for this project. Next step is writing the report.